# Full-text harvest of API-IL papers (run in Colab)
Downloads the **open-access full text** (Methods + tables) of API-IL papers from Europe PMC.
Abstracts often say only "eight new salts"; the full text names every salt. This only downloads - Claude does the reading.
Steps: Runtime > Run all. Takes ~20-40 min. At the end it downloads `api_il_fulltext.zip` - upload that zip to Claude.

In [ ]:
import requests, time, re, json, zipfile, io, pandas as pd
from xml.etree import ElementTree as ET
S=requests.Session(); S.headers['User-Agent']='DDP-IITM-research-harvest (contact: your email)'
BASE='https://www.ebi.ac.uk/europepmc/webservices/rest'

In [ ]:
KNOWN_PMIDS = ["42600255", "42364699", "42353752", "41754965", "41550903", "41290051", "41056550", "41028273", "40714163", "40697473", "40337224", "40320021", "40263323", "40257223", "40142964", "40062365", "39779636", "39654651", "39643195", "39491654", "39486779", "38590797", "38314703", "38271213", "38061517", "37984619", "37797342", "37442512", "37155370", "36910967", "36858176", "36806585", "36535965", "36430346", "36328197", "36320838", "36110370", "36014356", "35456711", "35449344", "34425167", "33989753", "33167474", "32852215", "32825444", "32391837", "31881750", "31763972", "31586728", "30735308", "29751143", "28199779", "28125212", "26617961", "26190053", "26070389", "25905624", "32261689", "24355776", "23978632", "23795613", "22534649", "20449254", "20143257", "42819875", "42822711", "42811607", "42725614", "42579116", "42559329", "42554322", "42518448", "42485080", "42446033", "42335254", "42316299", "42304734", "42300889", "42294038", "42261055", "42139351", "42137931", "42108822", "42091490", "41955073", "41935391", "41857894", "41769167", "41768748", "41690120", "41616984", "41572763", "41524294", "41478186", "41347960", "41294113", "41274078", "41213884", "41082503", "40934770", "40926978", "40787092", "40779899", "40779311", "40560061", "40521825", "40505917", "40485298", "40420471", "40364658", "40363569", "40349782", "40286214", "40270537", "40270476", "40230276", "40198870", "40192561", "40141184", "40076229", "40069217", "40064534", "40013328", "39933679", "39902693", "39791022", "39787781", "39778282", "39770062", "39634437", "39570941", "39566853", "39539601", "39458620", "39452425", "39339454", "39185607", "39139057", "39101858", "39032809", "38958607", "38882501", "38871169", "38823540", "38804505", "38794304", "38776534", "38773172", "38751336", "38744415", "38704382", "38694051", "38649746", "38629344", "38573315", "38427030", "38399436", "38385814", "38378468", "38279291", "38250392", "38193856", "38093637", "37755198", "37744857", "37606325", "37605522", "37570693", "37560895", "37552854", "37549381", "37529285", "37514120", "37499721", "37498344", "37367743", "37366119", "37346962", "37307801", "37262343", "37259417", "37163519", "37096366", "37042429", "36986721", "36964446", "36897097", "36874162", "36834945", "36736167", "36708328", "36674923", "36650743", "36643450", "36610512", "36599708", "36551498", "36548263", "36508194", "36235187", "36184454", "36171345", "36164963", "36058500", "36053025", "35974125", "35897904", "35854078", "35850530", "35833871", "35806581", "35803823", "35739370", "35697283", "35676441", "35670602", "35661084", "35658086", "35612374", "35612295", "35561456", "35549190", "35544610", "35499286", "35493141", "35479724", "35479433", "35417143", "35405279", "35389658", "35377144", "35352282", "35320751", "35213168", "35164193", "35110432", "35098237", "35085966", "35085728", "35054661", "35037708", "35011471", "34935795", "34896216", "34835626", "34795809", "34781526", "34771756", "34769430", "34766766", "34757348", "34601883", "34569591", "34520217", "34509733", "34501098", "34467755", "34389487", "34368549", "34358543", "34245140", "34229815", "34208767", "34206094", "34151080", "33977966", "33966177", "33899479", "33798684", "33639754", "33621293", "33581479", "33562597", "33556757", "33555100", "33504154", "33497566", "33491717", "33456012", "33435346", "33412875", "33403632", "33401768", "33253582", "33227209", "33191737", "33175246", "33147953", "33121087", "33118829", "33076658", "32981970", "32976927", "32974626", "32913906", "32902989", "32899785", "32899701", "32861769", "32853804", "32830497", "32775923", "35521105", "32727096", "32717808", "32717610", "32708674", "32633518", "32586055", "35517315", "32559699", "32545242", "32407287", "32363875", "32255269", "35498488", "35492138", "32213930", "32131540", "32057183", "31951238", "31950764", "31844602", "31842390", "31748056", "35541791", "31743529", "31690900", "31675893", "31668497", "31634657", "31621330", "31612799", "31593645", "31575176", "31568943", "31510743", "31508547", "31495176", "31460075", "31394882", "31326789", "31185691", "31163389", "31089620", "31083364", "35651142", "30966523", "30931300", "30909038", "30813375", "30740977", "30735216", "30679046", "30650210", "30622879", "30621273", "30596487", "30525630", "30501261", "30379539", "30362415", "30357184", "30307246", "30287015", "30271271", "30211149", "30198539", "30113612", "30094445", "30044658", "30036478", "29927596", "29874988", "29873442", "29843206", "29780992", "31458529", "29476561", "29470140", "29446859", "29441262", "29439903", "29347444", "29316116", "29290517", "29264915", "29259171", "29235721", "29233584", "29183116", "29097303", "29085931", "29063758", "29057996", "29039670", "28989662", "28976762", "28954512", "35650895", "28921788", "28901353", "28889750", "28883814", "28876934", "28849933", "28833758", "28786678", "28745056", "28715010", "28699755", "31457670", "31457654", "28653711", "28652593", "28650512", "28594004", "28574265", "28571336", "28540379", "28523100", "28471008", "28445028", "28423277", "28342356", "28321452", "30271684", "28244156", "28236691", "28189620", "28121151", "28091670", "27990721", "27935724", "27911045", "27870952", "27757446", "27734607", "27783527", "27783445", "27775202", "27661760", "27642262", "27554057", "27472069", "27428048", "27220984", "28773485", "27156087", "27136186", "26965316", "26945484", "26929087", "26908332", "26838377", "26800340", "26785216", "26660453", "26652691", "26549825", "30090234", "26509235", "26502775", "26384881", "26370450", "26346407", "26315452", "26282318", "26263086", "26186475", "26126126", "26101957", "26047302", "25985929", "25976317", "25973552", "25962152", "25923649", "25922888", "25907139", "25903898", "25811753", "25738418", "25723046", "25680201", "25648101", "25629464", "25580540", "25534684", "25372300", "25341189", "25269078", "25256477", "25158763", "25155840", "25151038", "25124894", "25062465", "25017178", "24998266", "24938933", "24919852", "24910062", "24909912", "24853329", "24795960", "24746413", "24733527", "24651922", "24644064", "24626335", "31986604", "24572273", "24561336", "24397562", "24225581", "24156286", "23986145", "23953616", "23873515", "23873270", "23868469", "23843268", "23788318", "23650965", "23642809", "23477603", "23452799", "23303733", "26291225", "23241082", "23214472", "23190351", "23180598", "23123180", "23098274", "23092400", "22967360", "22796026", "26291851", "22728356", "22690854", "22455028", "22455023", "22455019", "22455012", "22447692", "22431313", "22368114", "22336328", "22307263", "22268528", "22247701", "22236350", "22204280", "22201255", "22077384", "22032301", "22029384", "21858331", "21786823", "21769367", "21766952", "21755079", "21648424", "21601215", "21590825", "21566824", "21561120", "21557480", "21468445", "21428297", "21410213", "21405339", "21384444", "21341864", "21247213", "21230540", "21171617", "21159360", "21157603", "21079823", "21067238", "20949216", "20925318", "20888454", "20879719", "20863113", "20852767", "20707371", "20695502", "20694227", "20623574", "20572724", "20557035", "20532355", "20486149", "20423186", "20359243", "20235600", "20210074", "26613314", "20184304", "20170924", "20143862", "20135647", "20111792", "20004411", "19896645", "19879552", "19774567", "19758596", "19705891", "19566174", "19469567", "19209937", "19209929", "21706581", "19140627", "19071929", "19031387", "18969525", "18855466", "18854988", "18844423", "18758014", "18576455", "18092298", "18063302", "18044874", "17989797", "17949091", "17786916", "17725367", "17719598", "17691840", "17616224", "17574264", "17555633", "17509607", "17503808", "17429805", "17419609", "17340574", "17201441", "16851388", "16800602", "16524102", "15106966", "12924907", "12924549", "11429927", "42551677", "42390211", "41833418", "41738516", "40644785", "39515673", "39362445", "38532617", "38239439", "36903590", "36145678", "35456976", "35006819", "34207691", "33952857", "33895294", "33484731", "33189957", "32686889", "32109824", "35492154", "31923931", "31887061", "31761047", "35516352", "29302650", "29190552", "29129431", "28496471", "28254322", "27524004", "27420042", "26766632", "26080944", "25652555", "25618715", "40883933", "24853394", "24401404", "24381516", "23813951", "23587315", "23540249", "19650046", "19269644", "18440541", "15515047", "14680244", "41125746", "40987562", "40509289", "37178749", "34959392", "32750570", "32297419", "31862555", "31837407", "29576276", "29305055", "26627587", "24119759", "38812894", "19594143", "18632109", "18543868", "16860773", "41658092", "38675141", "32316479", "29929130", "24916075", "19674753", "41777839", "41707369", "38543216", "36297593", "31062890", "42289446", "38462106", "38183811", "37687045", "37187300", "37184044", "35887303", "34885965", "34695021", "31650263", "31167398", "30836180", "30470464", "30320287", "28240863", "27267864", "24411391", "23009988", "24364977", "21348304", "38831344", "36593986", "35806838", "35158186", "28986197", "24746353", "21561710", "42289611", "41754620", "41226164", "39864637", "39520352", "39069806", "36361664", "35011452", "34563532", "34452124", "32445198", "30955453", "30935036", "31457311", "28665154", "27865542", "27345738", "24865572", "21543269", "20145871", "17618639", "41678749", "40474289", "40454062", "40068713", "39987444", "39767048", "39275021", "39190253", "38957680", "38657919", "38434799", "37776933", "37057267", "36904316", "36731664", "36554442", "36530223", "35956933", "35666944", "34912003", "34392036", "33780694", "32905278", "32748878", "32096807", "31877863", "31642264", "31458372", "31218308", "30856442", "30471538", "29098337", "28390353", "28304017", "28275768", "27541982", "27178928", "26347332", "26318199", "25734818", "25281072", "24824879", "24615781", "24413748", "24329073", "24107181", "23910339", "23237715", "23062433", "22738425", "20032873", "20006071", "19201024", "17886840", "32396354", "41125729", "33073124", "32696844", "24900559", "23895461", "31986203", "26456294", "41931479", "41255933", "36903651", "36771044", "35101392", "33579445", "32314922", "26931706", "32261888", "25025600", "23742488", "23652993", "22713051", "22424553", "21161097", "30449072", "26286764", "15053682", "41096683", "39029636", "38474650", "35174380", "30690337", "26544047", "24239677", "21443681", "41459760", "33518856", "36521327", "32070532", "25039057", "24863203", "24615770", "18456946", "18342462", "17585777", "17082915", "41939378", "40042525", "39749771", "39033844", "38761622", "38157746", "38024745", "37520095", "36770117", "36696601", "34513622", "34139328", "36132906", "32531562", "32448888", "32196055", "32072997", "31679776", "31370238", "31036189", "30411483", "29505988", "29438870", "27998108", "27522461", "27474680", "26470029", "26418224", "25548871", "23021803", "21082676", "19268960", "42676129", "31129736", "30683382", "23842893", "22948804", "22213460", "21907341", "38707273", "33471500", "27769415", "27641624", "23740623", "23008691", "20981373", "38356122", "36095856", "31705607", "32769678", "36506216", "42556145", "41833083", "40891719", "32993877", "30709623", "30488649", "24296144", "21953836", "19632683", "41561008", "40674975", "33283901", "31602470", "27769476", "25271847", "23143005", "33168252", "26450655", "42429370", "41919210", "41762984", "41662939", "41045862", "40239518", "39997826", "38802523", "37922797", "37500441", "37481880", "36049348", "35112983", "34968035", "32224175", "31853645", "30319308", "29653419", "35542022", "28871466", "28490065", "27542460", "26871507", "24767438", "23450634", "23420737", "21268600", "42798879", "39293158", "39274924", "34974040", "35530033", "32344768", "32311469", "21657060", "20825949", "20813174", "20162145", "39970445", "40369026", "39822312", "39747962", "30414296", "28551136", "27662760", "29332813", "28142101", "24077084", "29403767", "29203112", "24016591", "41062607", "19042163", "35231956", "26755141", "18327676", "38372509", "38240794", "37526874", "36580766", "32823100", "32278420", "32060642", "31832629", "31704621", "30663787", "29438832", "24971654", "23598015", "22539321", "22265467", "16386264", "33562868", "27061492", "38566421", "31353071", "23312378", "25926028", "23303483", "22458964", "42331173", "32342529", "30294762", "28549907", "28118570", "25281143", "34110333", "32807914", "20210404", "35247446", "27836753", "41094024", "35546818", "21646436", "41455330", "35425111", "34146681", "31357362", "30981340", "27271823", "25498149", "34514237", "34451295", "17254370", "36368368", "31147154", "28971527", "30450594", "41788613", "41463313", "36531439", "32302369", "24394756", "24170554", "29462951", "39051346", "38677401", "37206222", "36770482", "34271081", "34209115", "32890655", "30439668", "42834667", "41723988", "39683899", "35456544", "33422736", "32373105", "29128705", "25499881", "34335913", "18291630", "36480787", "34882979", "31878190", "28733082", "42414423", "30303500", "27483222", "31877828", "42304047", "38789853", "33050373", "27914801", "23683867", "32782411", "33735710", "32247109", "31047752", "40142947", "39473626", "36458850", "34562557", "32025770", "30419220", "25400220", "24737623", "24107075", "41293939", "41243804", "39048011", "38742683", "38620030", "33533417", "32860147", "32067389", "31953082", "24668136", "24528755", "20553773", "37829321", "37110572", "36241668", "35006702", "34937015", "34522593", "32504973", "31437506", "27231909", "24322762", "23471455", "23134318", "41844026", "31991874", "28083577", "16364588", "41668786", "34834050", "32673948", "35529654", "30138820", "29508930", "28672835", "28230296", "27082812", "24913865", "36293152", "30771980", "30275632", "24070480", "39814203", "35057101", "42083560", "31396774", "23500478", "38319125", "35492502", "34364606", "35423994", "33422997", "33261106", "30569361", "27769445", "23375810", "40527702", "40445368", "37657924", "30083196", "29225645", "42826607", "41663445", "37569613", "33802422", "31500119", "30875750", "30856561", "35542470", "24790972", "18729115", "15516108", "36444862", "30271685", "38675157", "38396785", "38264907", "30319309", "38252882", "37959912", "36211776", "35425358", "33692237", "32722063", "30572636", "30563037", "30150809", "29860055", "27855896", "26194470", "25190186", "23861179", "23690034", "23598199", "21903215", "21250692", "20005511", "19818318", "19633814", "42142456", "39891662", "38965739", "37751672", "35863483", "35424928", "35638665", "30961967", "30622872", "29421039", "28045451", "26582151", "24583213", "24128399", "23799234", "23665696", "23380940", "22284874", "21909673", "21414442", "20559512", "41260171", "40637001", "42257191", "39083963", "38540944", "34240758", "34084078", "33107535", "32597002", "35514643", "30271267", "30028075", "28496974", "28027522", "26379471", "23627273", "30150588", "27980569", "26444144", "40392184", "39407468", "39310201", "37807721", "37765531", "36909746", "36014405", "30376336", "31962018", "37836601", "31436092", "35558303", "30147225", "29307532", "22747437", "20347908", "19640150", "26609613", "17915319", "17444674", "41984797", "20511289", "15010753", "42820471", "42809809", "42756738", "42741676", "42731556", "42729765", "42720200", "42687563", "42685877", "42673568", "42637886", "42629981", "42563245", "42504797", "42504456", "42494274", "42492710", "42489819", "42444479", "42443946", "42414084", "42383334", "42372833", "42366922", "42359820", "42359594", "42357397", "42310188", "42309201", "42255576", "42248286", "42231098", "42108052", "42105956", "42091492", "42074181", "42067046", "42066004", "41972773", "41908456", "41905421", "41900029", "41848665", "41848659", "41813676", "41786474", "41785559", "41744447", "41730660", "41702021", "41645705", "41639998", "41636450", "41632887", "41602181", "41561001", "41558558", "41552578", "41552507", "41537472", "41520696", "41502703", "41486744", "41481152", "41420246", "41375207", "41359980", "41330517", "41308765", "41240566", "41190387", "41187335", "41174442", "41110235", "41104461", "41088783", "41082287", "41074185", "41012230", "41011724", "40999743", "40985481", "40963478", "40960117", "40885359", "40827549", "40812156", "40804065", "40790007", "40778884", "40752689", "40727780", "40704282", "40693314", "40674782", "40669645", "40669616", "40665389", "40658448", "40621004", "40613602", "40605559", "40591733", "40576332", "40575930", "40536790", "40513066", "40492897", "40491251", "40481621", "40377918", "40371712", "40370844", "40366366", "40351005", "40333001", "40325345", "40310971", "40309118", "40297945", "40286162", "40255114", "40142056", "40132706", "40104838", "40098691", "40076402", "40076261", "40060006", "40057068", "40023982", "40018831", "40018296", "39996589", "39991822", "39958232", "39935958", "39914347", "39902108", "39854164", "39838768", "39828420", "39813775", "39764574", "39655517", "39603435", "39578478", "39545870", "39522923", "39468231", "39459184", "39442605", "39437967", "39427776", "39426355", "39398161", "39394262", "39391093", "39342670", "39341285", "39237671", "39236918", "40995071", "39170157", "39113547", "39094920", "39083218", "39035979", "38999093", "38959326", "38956391", "38772466", "38765482", "38723835", "38716816", "38697420", "38638223", "38611842", "38611804", "38607995", "38587168", "38574502", "38572510", "38545295", "38508073", "38426306", "38401717", "38399932", "38399102", "38369081", "38360538", "38346264", "38343917", "38335806", "38288905", "38262058", "38188979", "38141670", "38091907", "38075793", "38042312", "38029568", "38004535", "37999189", "37997686", "37994697", "37991202", "37985055", "37942450", "37909278", "37895973", "37894633", "37858290", "37836770", "37836024", "37787564", "37750571", "37699312", "37689282", "37633614", "37633544", "37611234", "37606049", "37576585", "37532059", "37513439", "37502854", "37501336", "37478610", "37470288", "37468651", "37462948", "37435408", "37415504", "37413969", "37407502", "37393574", "37374570", "37328015", "37327715", "37303163", "37272734", "37270075", "37265495", "37240166", "37200808", "37183344", "37177302", "37163669", "37119462", "37097129", "37067040", "37049892", "36960931", "36934812", "36925679", "36922947", "36903640", "36892140", "36863708", "36847770", "36824802", "36813330", "36736187", "36698308", "36651124", "36623577", "36622949", "36622301", "36565165", "36501542", "36499061", "36471542", "36402385", "36381128", "36359660", "36199602", "36186600", "36186557", "35968772", "35958129", "35956982", "35934167", "35908622", "35858287", "35848866", "35845396", "35835302", "35806268", "35761520", "35736287", "35661069", "35642388", "35631912", "35594750", "35585441", "35567816", "35545798", "35527836", "35500254", "35483639", "35478834", "35478777", "35468604", "35456612", "35450624", "35429708", "35425432", "35413771", "35413321", "35362493", "35262478", "35262105", "35197281", "35193052", "35182107", "35057053", "35049726", "35029253", "35002535", "34946635", "34897318", "34885786", "34783533", "34767843", "34746944", "34745654", "34729412", "34710329", "34577946", "34577069", "34460218", "34443929", "34423632", "34420733", "34418930", "34391350", "34361707", "34349928", "34337267", "34271329", "34268855", "34240972", "34163590", "34109780", "34050617", "34027084", "33919354", "33906032", "33885096", "33876098", "33803761", "33796955", "33737540", "33687390", "33685170", "33558588", "33554999", "33547738", "33440244", "33396610", "33393532", "33393288", "33373104", "33352431", "33340950", "33338943", "33287431", "33239302", "33205621", "33126573", "33110984", "33058352", "33007160", "32966447", "32887046", "32872533", "32839395", "32829807", "32820902", "32790388", "32786926", "35520061", "32735879", "32716215", "32674477", "32666802", "32589425", "32578619", "32544333", "32502857", "32412761", "32390824", "32380686", "32339547", "32286075", "32283021", "32272688", "32249857", "32232896", "32208319", "32201308", "32172563", "32141282", "32057854", "32033222", "31990285", "31972091", "31950705", "31936704", "31887855", "31882884", "31871754", "31867310", "31826452", "31800230", "31775594", "31767747", "31704086", "31687319", "31675504", "31672267", "31642611", "31603287", "31594975", "31570727", "31552929", "31465826", "31459182", "31405123", "31330104", "35519441", "33405745", "31278853", "31192327", "31176858", "31170494", "31137761", "31112631", "31087815", "35516334", "31075635", "35519561", "31059265", "31036163", "31005289", "30990712", "30965152", "30959998", "30939779", "30822737", "30811822", "30712349", "30682215", "30664995", "30648148", "30638804", "32254151", "30576736", "30475622", "30457137", "30455733", "30409002", "30404141", "30401741", "30398292", "30359021", "30347675", "30324359", "30307224", "30271272", "30260229", "30177986", "30169955", "30144546", "30134711", "30026081", "33435102", "29941553", "29892449", "29886224", "29878782", "29857229", "29744268", "29687272", "29671574", "29567497", "29534802", "35542011", "29449885", "29432015", "29425461", "29319089", "29279129", "29265818", "29239608", "29143022", "29131957", "29112809", "29075325", "29074204", "28962788", "28952726", "28936499", "28913646", "28842034", "28758402", "28703458", "28693317", "28659629", "28606518", "28597383", "28485942", "28480595", "28435968", "28418045", "28387515", "28337858", "28319840", "28111663", "27915199", "27809527", "27730734", "27773321", "27763737", "27722357", "27697607", "27550471", "27514793", "27503573", "27435197", "27386357", "27381851", "28773623", "27222006", "27188287", "27185089", "27179607", "27154683", "27072538", "26815603", "26743645", "29025362", "28330203", "26717817", "26661416", "26597485", "26540438", "26529242", "26395718", "26388068", "26322471", "26178665", "26093563", "26072299", "26040507", "26005926", "25966417", "25919083", "25908481", "25903143", "25898080", "25869239", "25868110", "25800513", "25798458", "25738708", "25694168", "25590423", "25460999", "25454813", "25454437", "25444662", "25433208", "25386894", "25386337", "25381609", "25380306", "25376613", "25277220", "25251987", "25242504", "25232641", "25157174", "25156719", "25127187", "25037344", "25013178", "24930496", "24874879", "24859211", "24802079", "24758931", "24745272", "24703634", "24699681", "24580821", "24507307", "24488260", "24399194", "24392011", "24323076", "24305780", "24268343", "24171395", "23915257", "23770800", "23758528", "23561571", "23541906", "23481471", "23352950", "24504148", "23197345", "23006550", "22995740", "22749125", "22744976", "22526779", "22524435", "22497364", "22456483", "22140677", "22089924", "22033601", "21909232", "21751777", "21739965", "21717582", "21709874", "21682314", "21565470", "21455538", "21332184", "21283901", "21249244", "21112808", "20835485", "20665687", "20438773", "20145855", "20145840", "34290544", "20024356", "19800875", "19523749", "19462080", "19458829", "19325790", "19243165", "19214321", "19139855", "19133739", "19089011", "19040254", "18956394", "17622408", "17580947", "17534999", "17388488", "16971989", "16878946", "16499344", "16156600", "15329717", "15281161", "15264850", "15116202", "14737527"]
print(len(KNOWN_PMIDS),'papers already found by the abstract harvest')

In [ ]:
def get(url, params=None, tries=4):
    for t in range(tries):
        try:
            r=S.get(url, params=params, timeout=60)
            if r.status_code==200: return r
            if r.status_code in (429,503): time.sleep(5*(t+1)); continue
            return None
        except Exception: time.sleep(5*(t+1))
    return None

def search_all(query, maxn=3000):
    out=[]; cursor='*'
    while True:
        r=get(BASE+'/search', dict(query=query, format='json', pageSize=1000, cursorMark=cursor, resultType='lite'))
        if r is None: break
        j=r.json(); res=j.get('resultList',{}).get('result',[])
        out+=res
        nc=j.get('nextCursorMark')
        if not res or nc==cursor or len(out)>=maxn: break
        cursor=nc; time.sleep(0.5)
    return out

In [ ]:
# 1) Map known PMIDs -> PMCID (open-access full text available?)
rec={}
for i in range(0,len(KNOWN_PMIDS),100):
    chunk=KNOWN_PMIDS[i:i+100]
    q='EXT_ID:('+' OR '.join(chunk)+') AND SRC:MED'
    for r in search_all(q, maxn=200):
        rec[r.get('pmid') or r['id']]=r
    time.sleep(0.5)
print(len(rec),'known papers found in Europe PMC')

# 2) NEW full-text searches (finds papers where the salt names are only in the body text)
FT_QUERIES=[
 '("ionic liquid" OR "ionic liquids") AND ("active pharmaceutical ingredient" OR "API-IL" OR "API-ILs" OR "drug-based ionic liquid") AND OPEN_ACCESS:y',
 '("organic salts and ionic liquids" OR "OSILs" OR "OSIL") AND OPEN_ACCESS:y',
 '("GUMBOS" OR "group of uniform materials based on organic salts") AND OPEN_ACCESS:y',
 '("protic ionic liquid" OR "protic ionic liquids") AND (drug OR pharmaceutical) AND OPEN_ACCESS:y',
 '("ionic liquid" OR "ionic liquids") AND ("drug salt" OR "lipophilic salt" OR "hydrophobic ion pair") AND OPEN_ACCESS:y',
 '("dual active" OR "dual-active" OR "double active") AND ("ionic liquid" OR "ionic liquids") AND OPEN_ACCESS:y',
 '("ionic liquid" OR "ionic liquids") AND ("transdermal" OR "topical") AND ("counterion" OR "counter-ion" OR "counter ion") AND OPEN_ACCESS:y',
 '("docusate" OR "lauryl sulfate" OR "dodecyl sulfate") AND ("ionic liquid" OR "lipophilic salt") AND (drug OR API) AND OPEN_ACCESS:y',
 '("choline" OR "cholinium") AND ("ionic liquid" OR "ionic liquids") AND ("drug" OR "active pharmaceutical") AND OPEN_ACCESS:y',
 '("amino acid ester" OR "amino acid alkyl ester") AND ("ionic liquid" OR "ionic liquids") AND OPEN_ACCESS:y',
]
newrec={}
for q in FT_QUERIES:
    res=search_all(q, maxn=3000)
    for r in res: newrec[r['id']+'|'+r.get('source','')]=r
    print(len(res),'hits :',q[:80])
allrec=list(rec.values())+list(newrec.values())
oa={}
for r in allrec:
    p=r.get('pmcid')
    if p: oa[p]=dict(pmcid=p,pmid=r.get('pmid',''),doi=r.get('doi',''),title=r.get('title',''),year=r.get('pubYear',''),journal=r.get('journalTitle',''))
print(len(oa),'papers with open-access full text to download')

In [ ]:
# 3) Download full text and keep text paragraphs + table rows
def text(e): return re.sub(r'\s+',' ',''.join(e.itertext())).strip()
KEEP=re.compile(r'ionic liquid|\bIL\b|\bILs\b|salt|\[[^\]]{1,40}\]\s*\[|ium\b|counter.?ion|synthes|prepared|neutraliz|metathesis|ion exchange|eutectic|proton transfer', re.I)
rows=[]; meta=[]
for k,(p,m) in enumerate(oa.items()):
    r=get(f'{BASE}/{p}/fullTextXML')
    if r is None: meta.append({**m,'got_fulltext':False}); continue
    try: root=ET.fromstring(r.content)
    except Exception: meta.append({**m,'got_fulltext':False}); continue
    meta.append({**m,'got_fulltext':True})
    body=root.find('.//body'); 
    for sec_i,sec in enumerate(root.iter('sec')):
        title=sec.find('title'); st=text(title) if title is not None else ''
        for para in sec.findall('p'):
            t=text(para)
            if KEEP.search(t): rows.append(dict(pmcid=p,kind='para',section=st[:120],text=t[:4000]))
    for tw in root.iter('table-wrap'):
        cap=tw.find('caption'); ct=text(cap) if cap is not None else ''
        for tr in tw.iter('tr'):
            cells=[text(c) for c in tr if c.tag in ('td','th')]
            rows.append(dict(pmcid=p,kind='table_row',section=ct[:200],text=' | '.join(cells)[:2000]))
    if k%50==0: print(k,'/',len(oa),'papers;',len(rows),'snippets')
    time.sleep(0.35)
FT=pd.DataFrame(rows); M=pd.DataFrame(meta)
print(M.got_fulltext.sum(),'full texts read;',len(FT),'snippets kept')

In [ ]:
buf='api_il_fulltext.zip'
with zipfile.ZipFile(buf,'w',zipfile.ZIP_DEFLATED) as z:
    z.writestr('fulltext_snippets.csv', FT.to_csv(index=False))
    z.writestr('papers.csv', M.to_csv(index=False))
from google.colab import files
files.download(buf)